In [1]:
from google.colab import drive

# Mount Google Drive to access the project files and datasets.
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import os
import re
from collections import Counter

# ============================================================
# PATHS
# ============================================================
FINAL_DATASET_DIR = "/content/drive/MyDrive/code_switch_project/data/final_dataset"
OUTPUT_DIR = "/content/drive/MyDrive/code_switch_project/data/augmented_dataset"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ============================================================
# LOAD SPLITS FROM FILE 02
# ============================================================
print("Loading train/validation/test splits...")
train_df = pd.read_csv(os.path.join(FINAL_DATASET_DIR, "train.csv"))
val_df = pd.read_csv(os.path.join(FINAL_DATASET_DIR, "validation.csv"))
test_df = pd.read_csv(os.path.join(FINAL_DATASET_DIR, "test.csv"))

print(f"Train: {len(train_df):,} rows")
print(f"Val: {len(val_df):,} rows")
print(f"Test: {len(test_df):,} rows")

# ============================================================
# AUGMENTATION PARAMETERS
# ============================================================
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

TRAIN_AUG_PROB = 0.40       # 40% of training rows get augmented
VAL_AUG_PROB = 0.25         # 25% of validation rows get augmented
KEEP_BOTH_PROB = 0.50       # 50% of augmented rows keep both original and augmented

Loading train/validation/test splits...
Train: 185,452 rows
Val: 39,740 rows
Test: 39,740 rows


In [3]:
# ============================================================
# ABBREVIATION DICTIONARIES (domain-specific)
# ============================================================
FR_ABBREVIATIONS = {
    'université': 'uni',
    'faculté': 'fac',
    'examen': 'exam',
    'rendez-vous': 'rdv',
    'préfecture': 'préf',
    'titre de séjour': 'tds',
    'formulaire': 'form',
    'inscription': 'inscri',
    'document': 'doc',
    'appartement': 'appart',
    'colocation': 'coloc',
    'propriétaire': 'proprio',
    'médecin': 'doc',
    'hôpital': 'hôpit',
    'pharmacie': 'pharma',
    'ordonnance': 'ordo',
    'travail': 'trav',
    'salaire': 'sal',
    'aujourd\'hui': 'auj',
    'toujours': 'tjrs',
    'peut-être': 'ptetre',
    'génial': 'genial',
    'sympathique': 'sympa',
    'restaurant': 'resto',
    'téléphone': 'tél',
    'portable': 'port',
    'message': 'msg',
    'application': 'app',
    'bonjour': 'bjr',
    'bonsoir': 'bsr',
    "s'il te plaît": 'stp',
    'mort de rire': 'mdr',
    'pété de rire': 'ptdr',
    'pourquoi': 'pk',
    'parce que': 'pcq',
    'quelqu\'un': 'qqn',
    'quelque chose': 'qqch',
    't\'inquiète': 'tkt',
    'désolé': 'dsl',
    'ça va': 'cv',
    'j\'ai': 'g',
    'je suis': 'jsuis',
    'je t\'aime': 'jtm',
    'à plus': 'a+',
    'ta gueule': 'tg',
    'wesh': 'wsh',
    'non': 'nn',
    'oui': 'ouii',
    'pour': 'pr',
    'vous': 'vs',
    'tout': 'tt',
    'beaucoup': 'bcp',
    'c\'est': 'c\'',
    'il y a': 'ya',
    'quoi': 'koi',
    'ne pas': 'pas',
}

DE_ABBREVIATIONS = {
    'Universität': 'uni',
    'Vorlesung': 'VL',
    'Semester': 'Sem',
    'Bibliothek': 'Bib',
    'Bürgeramt': 'BA',
    'Ausländerbehörde': 'ABH',
    'Formular': 'Form',
    'Dokument': 'Dok',
    'Wohnung': 'Whg',
    'liebe grüsse': 'lg',
    'viele grüsse': 'vg',
    'keine ahnung': 'ka',
    'kein plan': 'kp',
    'vielleicht': 'vlt',
    'irgendwie': 'iwie',
    'irgendwas': 'iwas',
    'bis bald': 'bb',
    'gute nacht': 'gn8',
    'hab dich ganz doll lieb': 'hdgdl',
    'zusammen': 'zsm',
    'okay': 'oke',
    'und': 'nd',
    'jetzt': 'jz',
    'eigentlich': 'eig',
    'kein plan warum': 'kp warum',
}

# ============================================================
# INFORMAL PARTICLES (language-specific filler words)
# ============================================================
FR_PARTICLES = ['genre', 'quoi', 'tu vois', 'tu sais', 'enfin', 'bah', 'ouais', 'mec', 'franchement']
DE_PARTICLES = ['halt', 'eben', 'ne', 'irgendwie', 'sowas', 'einfach', 'also', 'dann']

# ============================================================
# EMOJI MAPPING (keyword-based)
# ============================================================
EMOJI_KEYWORDS = {
    'université|fac|cours|exam|études|uni|schule|vorlesung': '🎓',
    'travail|job|boulot|emploi|stage|arbeit': '💼',
    'hôpital|médecin|pharmacie|urgence|krankenhaus|arzt': '🏥',
    'appartement|maison|wohnung|logement': '🏠',
    'merci|thank|danke|thanks': '😊',
    'non|nein|nicht|jamais|nie': '❌',
    'oui|ja|yes|accord|okay': '✅',
    'demain|morgen|bientôt|bald': '⏰',
    'police|polizei|agent': '🚔',
    'restaurant|café|food|essen': '🍽️',
}

# ============================================================
# AUGMENTATION PROBABILITIES (tunable parameters)
# ============================================================
ABBREV_PROB = 0.50          # 50% chance to apply abbreviations
PARTICLE_PROB = 0.15        # 15% chance to add informal particles
LOWERCASE_PROB = 0.10       # 10% chance to lowercase
PUNCT_PROB = 0.20           # 20% chance to remove trailing punctuation
EMOJI_PROB = 0.10           # 10% chance to add emoji

In [4]:
# ============================================================
# TRANSFORMATION FUNCTIONS
# ============================================================

def apply_abbreviations(text, abbrev_dict):
    """Replace words/phrases with their real slang abbreviations."""
    words_to_replace = [word for word in abbrev_dict.keys() if word.lower() in text.lower()]
    if not words_to_replace:
        return text

    to_replace = np.random.choice(
        words_to_replace,
        size=min(np.random.randint(1, 3), len(words_to_replace)),
        replace=False
    )

    result = text
    for word in to_replace:
        abbrev = abbrev_dict[word]
        result = re.sub(r'\b' + re.escape(word) + r'\b', abbrev, result, flags=re.IGNORECASE)

    return result

def add_informal_particle(text, particles):
    """Add 1 informal particle to the sentence (10-15% chance)."""
    # Only apply to sentences with at least 4 words
    word_count = len(text.split())
    if word_count < 4 or np.random.random() > 0.125:  # 12.5% chance (middle of 10-15% range)
        return text

    particle = np.random.choice(particles)
    # Add before punctuation if exists
    if text.endswith(('.', '!', '?')):
        return text[:-1] + f", {particle}" + text[-1]
    else:
        return text + f", {particle}"

def remove_trailing_punctuation(text):
    """Remove trailing punctuation."""
    return text.rstrip('.!?,;:')

def add_emoji(text):
    """Add emoji based on keywords in text."""
    text_lower = text.lower()
    for keywords, emoji in EMOJI_KEYWORDS.items():
        if any(re.search(kw, text_lower) for kw in keywords.split('|')):
            if np.random.random() > 0.5:  # 50% chance to add if keyword matches
                return text + f" {emoji}"
    return text

def lowercase_randomly(text):
    """Convert to lowercase."""
    return text.lower()

In [5]:
# ============================================================
# AUGMENTATION PIPELINE
# ============================================================

def augment_sentence(text, language):
    """
    Apply 1-3 randomly selected transformations to a sentence.
    The language determines which language-specific abbreviations
    and informal particles are used.
    """

    transformations_applied = []

    available_transformations = [
        'abbrev',
        'particle',
        'lowercase',
        'punct',
        'emoji'
    ]

    # Randomly select between 1 and 3 different transformations.
    n_transformations = np.random.randint(1, 4)

    selected_transformations = np.random.choice(
        available_transformations,
        size=n_transformations,
        replace=False
    )

    for transformation in selected_transformations:

        original_text = text

        # -------------------------
        # Abbreviation transformation
        # -------------------------
        if transformation == 'abbrev':

            if language == 'french':
                text = apply_abbreviations(
                    text,
                    FR_ABBREVIATIONS
                )

            elif language == 'german':
                text = apply_abbreviations(
                    text,
                    DE_ABBREVIATIONS
                )

            elif language == 'mixed':
                # Randomly use French or German abbreviations.
                if np.random.random() < 0.5:
                    text = apply_abbreviations(
                        text,
                        FR_ABBREVIATIONS
                    )
                else:
                    text = apply_abbreviations(
                        text,
                        DE_ABBREVIATIONS
                    )

        # -------------------------
        # Informal particle transformation
        # -------------------------
        elif transformation == 'particle':

            if language == 'french':
                text = add_informal_particle(
                    text,
                    FR_PARTICLES
                )

            elif language == 'german':
                text = add_informal_particle(
                    text,
                    DE_PARTICLES
                )

            elif language == 'mixed':
                # Randomly use a French or German particle.
                if np.random.random() < 0.5:
                    text = add_informal_particle(
                        text,
                        FR_PARTICLES
                    )
                else:
                    text = add_informal_particle(
                        text,
                        DE_PARTICLES
                    )

        # -------------------------
        # Lowercase transformation
        # -------------------------
        elif transformation == 'lowercase':
            text = lowercase_randomly(text)

        # -------------------------
        # Punctuation transformation
        # -------------------------
        elif transformation == 'punct':
            text = remove_trailing_punctuation(text)

        # -------------------------
        # Emoji transformation
        # -------------------------
        elif transformation == 'emoji':
            text = add_emoji(text)

        # Record only transformations that actually changed the text.
        if text != original_text:
            transformations_applied.append(transformation)

    return text, transformations_applied


# ============================================================
# DATASET AUGMENTATION FUNCTION
# ============================================================

def augment_dataset(df, augmentation_probability, keep_both_probability):
    """
    Apply augmentation to selected rows of a dataset.

    augmentation_probability:
        Probability that a row is selected for augmentation.

    keep_both_probability:
        Probability of keeping both the original and augmented
        versions when augmentation produces a change.
    """

    augmented_rows = []

    stats = {
        'original': 0,
        'replaced': 0,
        'kept_both': 0,
        'no_change': 0
    }

    for idx, row in df.iterrows():

        original_row = row.to_dict()

        # Keep the original row by default.
        augmented_rows.append(original_row)
        stats['original'] += 1

        # Randomly decide whether this row should be augmented.
        if np.random.random() < augmentation_probability:

            augmented_text, transformations = augment_sentence(
                row['text'],
                row['language']
            )

            # If the transformations produced no actual change,
            # keep the original sentence unchanged.
            if augmented_text == row['text']:

                stats['no_change'] += 1
                continue

            # Keep both the original and augmented versions.
            if np.random.random() < keep_both_probability:

                augmented_row = row.to_dict()
                augmented_row['text'] = augmented_text

                augmented_rows.append(augmented_row)
                stats['kept_both'] += 1

            # Otherwise replace the original with the augmented version.
            else:

                augmented_rows[-1]['text'] = augmented_text
                stats['replaced'] += 1

    return pd.DataFrame(augmented_rows), stats


# ============================================================
# AUGMENT TRAINING SET
# ============================================================

print("\n" + "="*60)
print("AUGMENTING TRAINING SET")
print("="*60)

train_augmented_df, train_stats = augment_dataset(
    train_df,
    TRAIN_AUG_PROB,
    KEEP_BOTH_PROB
)

print(f"Original rows: {train_stats['original']:,}")
print(f"Replaced rows: {train_stats['replaced']:,}")
print(f"Kept both: {train_stats['kept_both']:,}")
print(f"No effective change: {train_stats['no_change']:,}")
print(f"Final training size: {len(train_augmented_df):,}")


# ============================================================
# AUGMENT VALIDATION SET
# ============================================================

print("\n" + "="*60)
print("AUGMENTING VALIDATION SET")
print("="*60)

val_augmented_df, val_stats = augment_dataset(
    val_df,
    VAL_AUG_PROB,
    KEEP_BOTH_PROB
)

print(f"Original rows: {val_stats['original']:,}")
print(f"Replaced rows: {val_stats['replaced']:,}")
print(f"Kept both: {val_stats['kept_both']:,}")
print(f"No effective change: {val_stats['no_change']:,}")
print(f"Final validation size: {len(val_augmented_df):,}")


# ============================================================
# CREATE CLEAN AND AUGMENTED TEST SETS
# ============================================================

print("\n" + "="*60)
print("CREATING TEST SET VERSIONS")
print("="*60)

# Keep the original test set unchanged for standard evaluation.
test_clean_df = test_df.copy()

print(f"Clean test size: {len(test_clean_df):,}")


# Create a separate augmented version for robustness evaluation.
test_augmented_rows = []

for _, row in test_df.iterrows():

    augmented_text, _ = augment_sentence(
        row['text'],
        row['language']
    )

    augmented_row = row.to_dict()
    augmented_row['text'] = augmented_text

    test_augmented_rows.append(augmented_row)


test_augmented_df = pd.DataFrame(test_augmented_rows)

print(f"Augmented test size: {len(test_augmented_df):,}")


print("\n" + "="*60)
print("AUGMENTATION COMPLETE")
print("="*60)


AUGMENTING TRAINING SET
Original rows: 185,452
Replaced rows: 26,798
Kept both: 26,570
No effective change: 20,690
Final training size: 212,022

AUGMENTING VALIDATION SET
Original rows: 39,740
Replaced rows: 3,562
Kept both: 3,484
No effective change: 2,796
Final validation size: 43,224

CREATING TEST SET VERSIONS
Clean test size: 39,740
Augmented test size: 39,740

AUGMENTATION COMPLETE


In [6]:
# ============================================================
# SAVE AUGMENTED DATASETS
# ============================================================

print("Saving augmented datasets...")


# Save augmented training set
train_output_path = os.path.join(
    OUTPUT_DIR,
    "train_augmented.csv"
)

train_augmented_df.to_csv(
    train_output_path,
    index=False
)

print(f"✓ Training set saved: {train_output_path}")
print(f"  Rows: {len(train_augmented_df):,}")


# Save augmented validation set
val_output_path = os.path.join(
    OUTPUT_DIR,
    "val_augmented.csv"
)

val_augmented_df.to_csv(
    val_output_path,
    index=False
)

print(f"✓ Validation set saved: {val_output_path}")
print(f"  Rows: {len(val_augmented_df):,}")


# Save the original, clean test set
test_clean_path = os.path.join(
    OUTPUT_DIR,
    "test_clean.csv"
)

test_clean_df.to_csv(
    test_clean_path,
    index=False
)

print(f"✓ Test set (clean) saved: {test_clean_path}")
print(f"  Rows: {len(test_clean_df):,}")


# Save the augmented test set separately
test_augmented_path = os.path.join(
    OUTPUT_DIR,
    "test_augmented.csv"
)

test_augmented_df.to_csv(
    test_augmented_path,
    index=False
)

print(f"✓ Test set (augmented) saved: {test_augmented_path}")
print(f"  Rows: {len(test_augmented_df):,}")


# ============================================================
# SUMMARY REPORT
# ============================================================

summary = f"""
{'='*60}
AUGMENTATION PIPELINE SUMMARY
{'='*60}

TRAINING SET:
  Original rows: {len(train_df):,}
  Final rows: {len(train_augmented_df):,}
  Data multiplication: {len(train_augmented_df) / len(train_df):.2f}x

VALIDATION SET:
  Original rows: {len(val_df):,}
  Final rows: {len(val_augmented_df):,}
  Data multiplication: {len(val_augmented_df) / len(val_df):.2f}x

TEST SET (TWO VERSIONS):
  Clean test: {len(test_clean_df):,} rows
  Augmented test: {len(test_augmented_df):,} rows

AUGMENTATION PARAMETERS:
  Train augmentation probability: {TRAIN_AUG_PROB * 100:.0f}%
  Val augmentation probability: {VAL_AUG_PROB * 100:.0f}%
  Keep both probability: {KEEP_BOTH_PROB * 100:.0f}%

TRANSFORMATION PROBABILITIES:
  Abbreviations: {ABBREV_PROB * 100:.0f}%
  Particles: {PARTICLE_PROB * 100:.0f}%
  Lowercase: {LOWERCASE_PROB * 100:.0f}%
  Remove punctuation: {PUNCT_PROB * 100:.0f}%
  Emoji: {EMOJI_PROB * 100:.0f}%

TOTAL DATASET SIZE:
  Train + Val + Test (clean + augmented): {
      len(train_augmented_df)
      + len(val_augmented_df)
      + len(test_clean_df)
      + len(test_augmented_df)
  :,} rows

All datasets saved to: {OUTPUT_DIR}
{'='*60}
"""

print(summary)


# Save the summary report as a text file
summary_path = os.path.join(
    OUTPUT_DIR,
    "augmentation_summary.txt"
)

with open(summary_path, 'w') as f:
    f.write(summary)

print(f"\n✓ Summary saved to: {summary_path}")

Saving augmented datasets...
✓ Training set saved: /content/drive/MyDrive/code_switch_project/data/augmented_dataset/train_augmented.csv
  Rows: 212,022
✓ Validation set saved: /content/drive/MyDrive/code_switch_project/data/augmented_dataset/val_augmented.csv
  Rows: 43,224
✓ Test set (clean) saved: /content/drive/MyDrive/code_switch_project/data/augmented_dataset/test_clean.csv
  Rows: 39,740
✓ Test set (augmented) saved: /content/drive/MyDrive/code_switch_project/data/augmented_dataset/test_augmented.csv
  Rows: 39,740

AUGMENTATION PIPELINE SUMMARY

TRAINING SET:
  Original rows: 169,638
  Final rows: 212,022
  Data multiplication: 1.25x

VALIDATION SET:
  Original rows: 36,351
  Final rows: 43,224
  Data multiplication: 1.19x

TEST SET (TWO VERSIONS):
  Clean test: 39,740 rows
  Augmented test: 39,740 rows

AUGMENTATION PARAMETERS:
  Train augmentation probability: 40%
  Val augmentation probability: 25%
  Keep both probability: 50%

TRANSFORMATION PROBABILITIES:
  Abbreviations: 